# Day 28: Exercises - Data Leakage and Overfitting

In this exercise, we will:
1. **Hunt for Data Leakage:** Show how scaling the entire dataset before splitting causes information from the validation set to leak into the training process.
2. **Detect Overfitting:** Train a highly complex model (like an unconstrained Random Forest) and observe the gap between training and validation metrics.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Load data
california = fetch_california_housing(as_frame=True)
X = california.data
y = california.target

print(f"Dataset shape: {X.shape}")

Dataset shape: (20640, 8)


## 1. Data Leakage

**The WRONG Way (Leakage):** Scaling the entire dataset `X` *before* splitting it. This means the mean and variance used for scaling are influenced by the validation data.

In [2]:
# WRONG: Scaling before splitting
scaler_wrong = StandardScaler()
X_scaled_wrong = scaler_wrong.fit_transform(X)

# Now splitting the already scaled data
X_train_wrong, X_val_wrong, y_train, y_val = train_test_split(X_scaled_wrong, y, test_size=0.2, random_state=42)

print("Notice how the scaler has 'seen' the validation data during fit_transform(). This is Data Leakage!")

Notice how the scaler has 'seen' the validation data during fit_transform(). This is Data Leakage!


**The RIGHT Way (No Leakage):** Split first, then fit the scaler ONLY on the training data. Use that fitted scaler to transform the validation data.

In [3]:
# RIGHT: Split first
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit scaler only on training data
scaler_right = StandardScaler()
X_train_scaled = scaler_right.fit_transform(X_train)

# Transform validation data using the scaler fitted on training data
X_val_scaled = scaler_right.transform(X_val)

print("No leakage! The validation data remained completely unseen during the scaling process.")

No leakage! The validation data remained completely unseen during the scaling process.


## 2. Detecting Overfitting

We will train an unconstrained Random Forest (which naturally grows very deep trees) and compare its performance on the train vs. validation set.

In [4]:
# Train an unconstrained Random Forest
rf_overfit = RandomForestRegressor(random_state=42, n_estimators=50, max_depth=None)
rf_overfit.fit(X_train_scaled, y_train)

# Predictions
y_train_pred = rf_overfit.predict(X_train_scaled)
y_val_pred = rf_overfit.predict(X_val_scaled)

# Evaluate
train_r2 = r2_score(y_train, y_train_pred)
val_r2 = r2_score(y_val, y_val_pred)

print(f"Train R2: {train_r2:.4f}")
print(f"Val R2:   {val_r2:.4f}")
print(f"Gap:      {train_r2 - val_r2:.4f}")
print("\nLook at that huge gap! The model memorized the training data (near perfect R2) but doesn't generalize as perfectly. This is overfitting.")

Train R2: 0.9722
Val R2:   0.8042
Gap:      0.1680

Look at that huge gap! The model memorized the training data (near perfect R2) but doesn't generalize as perfectly. This is overfitting.


**Mitigating Overfitting:** Let's constrain the depth of the trees using `max_depth=5`.

In [5]:
# Train a constrained Random Forest
rf_constrained = RandomForestRegressor(random_state=42, n_estimators=50, max_depth=5)
rf_constrained.fit(X_train_scaled, y_train)

# Predictions
y_train_pred_c = rf_constrained.predict(X_train_scaled)
y_val_pred_c = rf_constrained.predict(X_val_scaled)

# Evaluate
train_r2_c = r2_score(y_train, y_train_pred_c)
val_r2_c = r2_score(y_val, y_val_pred_c)

print(f"Train R2 (constrained): {train_r2_c:.4f}")
print(f"Val R2   (constrained): {val_r2_c:.4f}")
print(f"Gap:                    {train_r2_c - val_r2_c:.4f}")
print("\nThe gap is much smaller now! The model is more robust and less prone to overfitting.")

Train R2 (constrained): 0.6814
Val R2   (constrained): 0.6469
Gap:                    0.0346

The gap is much smaller now! The model is more robust and less prone to overfitting.
